In [ ]:
# Copyright 2026 International Business Machines

# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at

#     http://www.apache.org/licenses/LICENSE-2.0

# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# 1.1 Introduction to EO Foundation Models

## Fine-Tuning with TerraMind

---

## 📚 Workshop Overview

This workshop introduces participants to **Earth Observation (EO) foundation models** and provides hands-on experience with fine-tuning and inference workflows.

In this notebook you will work with [TerraMind](https://huggingface.co/ibm-esa-geospatial) to:

- Understand EO foundation model concepts
- Perform a simple fine-tuning workflow 
- Run inference with pre-trained TerraMind models
- Compare single-modal (Sentinel-2) vs multi-modal (RGB+DEM) predictions

The session builds practical intuition for how EO foundation models are adapted to downstream tasks.  
  
Please see the Hugging Face page for more details of the saltmarsh models used in this session.

---

## 📋 Prerequisites

Please read the [instructions](https://github.com/IBM/ML4EO-workshop-2026) on the repository landing page.

---

## 0. Setup

### 0.1 Check Python Version

It's recommended that you run this notebook using Python 3.12.

In [ ]:
!python --version

### 0.2 Install and Import Dependencies

If running locally, ensure you have the necessary packages installed. This assumes you're in the project root directory.

Once installed, import dependencies.

In [ ]:
# Install dependencies TODO: ADD REMAINING
import sys
if 'google.colab' in sys.modules:
  !uv pip install --system gdown huggingface_hub
else:
  !pip install uv
  !uv pip install gdown huggingface_hub

In [1]:
import os
import warnings
from pathlib import Path
from huggingface_hub import hf_hub_download

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

print("✓ Dependencies imported successfully!")

✓ Dependencies imported successfully!


### 0.3 Download Sample Dataset

Before proceeding, you'll need to download the **sample dataset** for fine-tuning.

The dataset will be downloaded from Google Drive as a zip file and extracted to the `data/` directory.

Expected structure after extraction:
```
data/saltmarsh_sample/
  ├── train/
  │   ├── images/
  │   └── labels/
  ├── val/
  │   ├── images/
  │   └── labels/
  └── test/
      ├── images/
      └── labels/
```

In [2]:
import gdown
import zipfile

# TODO: Replace with actual Google Drive file ID
file_id = "YOUR_GOOGLE_DRIVE_FILE_ID_HERE"

# Download location
zip_output = "saltmarsh_sample.zip"
extract_to = "../../data"

# Download the zip file from Google Drive
if not os.path.isfile(zip_output):
    print("Downloading dataset from Google Drive...")
    gdown.download(
        f"https://drive.google.com/uc?id={file_id}",
        zip_output,
        quiet=False
    )
    print(f"✓ Downloaded to {zip_output}")
else:
    print(f"✓ Zip file already exists: {zip_output}")

# Extract the zip file
if os.path.isfile(zip_output):
    print(f"\nExtracting to {extract_to}...")
    with zipfile.ZipFile(zip_output, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
    print("✓ Extraction complete!")
    
    # Verify the structure
    data_path = Path(extract_to) / "saltmarsh_sample"
    if data_path.exists():
        print(f"\n✓ Dataset ready at: {data_path}")
        print("\nDataset structure:")
        for split in ['train', 'val', 'test']:
            split_path = data_path / split
            if split_path.exists():
                print(f"  {split}/")
                for subdir in ['images', 'labels']:
                    subdir_path = split_path / subdir
                    if subdir_path.exists():
                        count = len(list(subdir_path.glob('*')))
                        print(f"    {subdir}/ ({count} files)")
    else:
        print(f"⚠ Warning: Expected directory not found: {data_path}")
else:
    print("⚠ Download failed. Please check the file_id and try again.")

FileURLRetrievalError: Failed to retrieve file url:

	Cannot retrieve the public link of the file. You may need to change
	the permission to 'Anyone with the link', or have had many accesses.
	Check FAQ in https://github.com/wkentaro/gdown?tab=readme-ov-file#faq.

You may still be able to access the file from the browser:

	https://drive.google.com/uc?id=YOUR_GOOGLE_DRIVE_FILE_ID_HERE

but Gdown can't. Please check connections and permissions.

### 0.4 Set Up Paths for Fine-Tuning

Define the paths we'll need for fine-tuning and testing.

In [3]:
# Project root (adjust if running from different location)
project_root = Path("../..").resolve()
print(f"Project root: {project_root}")

# Local config file for fine-tuning (in repo)
config_file = project_root / "config/config-ibm-geospatial-saltmarsh-uk-s2-extent.yaml"

# Data paths (sample dataset)
data_root = project_root / "data/saltmarsh_sample"
train_images = data_root / "train/images"
train_labels = data_root / "train/labels"
val_images = data_root / "val/images"
val_labels = data_root / "val/labels"
test_images = data_root / "test/images"
test_labels = data_root / "test/labels"

# Output directory for fine-tuning
output_dir = project_root / "data/finetune/s2"
os.makedirs(output_dir, exist_ok=True)

print(f"\n✓ Paths configured for fine-tuning:")
print(f"  Config file: {config_file}")
print(f"  Data root: {data_root}")
print(f"  Output directory: {output_dir}")

Project root: /Users/annejones/old-macbook/ibm-repos/ML4EO-workshop-2026

✓ Paths configured for fine-tuning:
  Config file: /Users/annejones/old-macbook/ibm-repos/ML4EO-workshop-2026/config/config-ibm-geospatial-saltmarsh-uk-s2-extent.yaml
  Data root: /Users/annejones/old-macbook/ibm-repos/ML4EO-workshop-2026/data/saltmarsh_sample
  Output directory: /Users/annejones/old-macbook/ibm-repos/ML4EO-workshop-2026/data/finetune/s2


---

## 1. Fine-Tuning

### 1.1 Introduction to Fine-Tuning

Fine-tuning is the process of adapting a pre-trained foundation model to a specific task or domain. This approach leverages the knowledge learned during pre-training while specializing the model for your use case, enabling a task-specific model to be created.


In this section, we'll fine-tune a TerraMind model on salt marsh extent detection using Sentinel-2 imagery.

### 1.2 Prepare Fine-Tuning Command

We'll use TerraTorch's command-line interface to fine-tune the model. The configuration file (local, in the repo) specifies all the training parameters.

For this demo, we'll:
- Train for **1 epoch** only
- Use a **batch size of 2** for faster execution
- Use the **sample dataset** (smaller subset for demonstration)

The model will be trained using the local config file which contains all the hyperparameters and model architecture details.

In [ ]:
# Construct the fine-tuning command
fine_tuning_command = f"""terratorch fit \
    --config {config_file} \
    --trainer.default_root_dir {output_dir} \
    --trainer.max_epochs 1 \
    --data.init_args.batch_size 2"""

print("Fine-tuning command:")
print(fine_tuning_command)
print("\nNote: Using max_epochs=1 and batch_size=2 for quick demo.")
print("For full training, increase epochs and adjust batch size based on available GPU memory.")

### 1.3 Run Fine-Tuning

Execute the fine-tuning command. This will train the model for 1 epoch and save checkpoints to the output directory.

**Note:** TerraTorch (built on PyTorch Lightning) automatically creates the directory structure:
- `lightning_logs/` - Training logs directory
- `version_0/` - First run (increments for subsequent runs)
- `checkpoints/` - Contains the `.ckpt` checkpoint files

In [ ]:
# Run fine-tuning
!{fine_tuning_command}

### 1.4 Locate the New Checkpoint

After training, let's find the checkpoint that was just created.

In [ ]:
# Find the newly created checkpoint
new_checkpoint_dir = output_dir / "lightning_logs/version_0/checkpoints"
if new_checkpoint_dir.exists():
    new_checkpoints = list(new_checkpoint_dir.glob("*.ckpt"))
    if new_checkpoints:
        print(f"✓ Found {len(new_checkpoints)} checkpoint(s):")
        for ckpt in new_checkpoints:
            print(f"  - {ckpt.name}")
        # Use the first checkpoint for subsequent steps if needed
        new_checkpoint_path = new_checkpoints[0]
        print(f"\nUsing checkpoint: {new_checkpoint_path}")
    else:
        print("⚠ No checkpoints found in the directory.")
else:
    print(f"⚠ Checkpoint directory not found: {new_checkpoint_dir}")

---

## 2. Testing

### 2.1 Run Test on Fine-Tuned Model

Now we'll test the model we just fine-tuned on the test dataset.

This will evaluate the model's performance on held-out test data and report metrics.

In [ ]:
# Construct the test command using the newly fine-tuned checkpoint
test_command = f"""terratorch test \
    --config {config_file} \
    --ckpt_path {new_checkpoint_path} \
    --trainer.default_root_dir {output_dir}"""

print("Test command:")
print(test_command)

In [ ]:
# Run testing
!{test_command}

---

## 3. Inference with Pre-Trained Models

Now we'll download pre-trained models from HuggingFace and run inference to compare different approaches.

### 3.1 Download Pre-Trained Checkpoints from HuggingFace

We'll download two pre-trained models:
1. **S2 Extent Model** - Sentinel-2 based salt marsh extent detection
2. **RGB+DEM Extent Model** - Multi-modal (aerial RGB + DEM) salt marsh extent detection

In [6]:
# Import helper function for downloading from HuggingFace
from helper import download_model_from_hf

# HuggingFace repository information
s2_repo_id = "rosielickorish/ibm-geospatial-saltmarsh-uk-s2-extent-10m"
s2_config_name = "config-ibm-geospatial-saltmarsh-uk-s2-extent.yaml"
s2_checkpoint_name = "ibm-geospatial-saltmarsh-uk-s2-extent-10m_state_dict.ckpt"

rgbdem_repo_id = "rosielickorish/ibm-geospatial-saltmarsh-uk-rgbdem-extent-2m"
rgbdem_config_name = "config-ibm-geospatial-saltmarsh-uk-rgbdem-extent-2m.yaml"
rgbdem_checkpoint_name = "ibm-geospatial-saltmarsh-uk-rgbdem-extent-2m_state_dict.ckpt"

print("Downloading S2 model from HuggingFace...")
s2_config_path, s2_checkpoint_path = download_model_from_hf(
    repo_id=s2_repo_id,
    config_name=s2_config_name,
    checkpoint_name=s2_checkpoint_name,
    project_root=project_root
)
print(f"✓ S2 config: {s2_config_path}")
print(f"✓ S2 checkpoint: {s2_checkpoint_path}")

print("\nDownloading RGB+DEM model from HuggingFace...")
rgbdem_config_path, rgbdem_checkpoint_path = download_model_from_hf(
    repo_id=rgbdem_repo_id,
    config_name=rgbdem_config_name,
    checkpoint_name=rgbdem_checkpoint_name,
    project_root=project_root
)
print(f"✓ RGB+DEM config: {rgbdem_config_path}")
print(f"✓ RGB+DEM checkpoint: {rgbdem_checkpoint_path}")

print("\n✓ All models downloaded successfully!")

(…)m-geospatial-saltmarsh-uk-s2-extent.yaml:   0%|          | 0.00/3.14k [00:00<?, ?B/s]

ibm-geospatial-saltmarsh-uk-s2-extent-10(…):   0%|          | 0.00/363M [00:00<?, ?B/s]

✓ S2 config: /Users/annejones/old-macbook/ibm-repos/ML4EO-workshop-2026/configs/inference/config-ibm-geospatial-saltmarsh-uk-s2-extent.yaml
✓ S2 checkpoint: /Users/annejones/old-macbook/ibm-repos/ML4EO-workshop-2026/data/checkpoints/ibm-geospatial-saltmarsh-uk-s2-extent-10m_state_dict.ckpt



(…)atial-saltmarsh-uk-rgbdem-extent-2m.yaml:   0%|          | 0.00/2.82k [00:00<?, ?B/s]

ibm-geospatial-saltmarsh-uk-rgbdem-exten(…):   0%|          | 0.00/362M [00:00<?, ?B/s]

✓ RGB+DEM config: /Users/annejones/old-macbook/ibm-repos/ML4EO-workshop-2026/configs/inference/config-ibm-geospatial-saltmarsh-uk-rgbdem-extent-2m.yaml
✓ RGB+DEM checkpoint: /Users/annejones/old-macbook/ibm-repos/ML4EO-workshop-2026/data/checkpoints/ibm-geospatial-saltmarsh-uk-rgbdem-extent-2m_state_dict.ckpt

✓ All models downloaded successfully!


### 3.2 Download Inference Data

Download the inference dataset from Google Drive. This contains sample images for running inference with both S2 and RGB+DEM models.

Expected structure after extraction:
```
data/inference/
  └── images/
      ├── s2/          # Sentinel-2 imagery
      └── aerial/      # RGB + DEM aerial imagery
```

In [ ]:
import gdown
import zipfile

# TODO: Replace with actual Google Drive file ID for inference data
inference_file_id = "YOUR_INFERENCE_DATA_FILE_ID_HERE"

# Download location
inference_zip_output = "inference_data.zip"
inference_extract_to = "../../data"

# Download the zip file from Google Drive
if not os.path.isfile(inference_zip_output):
    print("Downloading inference data from Google Drive...")
    gdown.download(
        f"https://drive.google.com/uc?id={inference_file_id}",
        inference_zip_output,
        quiet=False
    )
    print(f"✓ Downloaded to {inference_zip_output}")
else:
    print(f"✓ Zip file already exists: {inference_zip_output}")

# Extract the zip file
if os.path.isfile(inference_zip_output):
    print(f"\nExtracting to {inference_extract_to}...")
    with zipfile.ZipFile(inference_zip_output, 'r') as zip_ref:
        zip_ref.extractall(inference_extract_to)
    print("✓ Extraction complete!")
    
    # Verify the structure
    inference_data_path = Path(inference_extract_to) / "inference"
    if inference_data_path.exists():
        print(f"\n✓ Inference data ready at: {inference_data_path}")
        print("\nInference data structure:")
        images_path = inference_data_path / "images"
        if images_path.exists():
            for modality in ['s2', 'aerial']:
                modality_path = images_path / modality
                if modality_path.exists():
                    count = len(list(modality_path.glob('*')))
                    print(f"  images/{modality}/ ({count} files)")
    else:
        print(f"⚠ Warning: Expected directory not found: {inference_data_path}")
else:
    print("⚠ Download failed. Please check the file_id and try again.")

### 3.3 Set Up Inference Paths

Define paths for inference outputs and data.

In [5]:
# Checkpoint paths
s2_checkpoint_path = Path(s2_checkpoint_local)
rgbdem_checkpoint_path = Path(rgbdem_checkpoint_local)

# Base inference directory
inference_base = project_root / "data/inference"

# Inference data paths - organized by data type
inference_images_s2 = inference_base / "images/s2"
inference_images_aerial = inference_base / "images/aerial"
inference_images_dem = inference_base / "images/dem"

# Label paths - organized by model type
inference_labels_s2 = inference_base / "labels/S2"
inference_labels_rgbdem = inference_base / "labels/rgb_dem_extent"

# Prediction output paths - organized by model type
s2_inference_output = inference_base / "pred/S2"
rgbdem_inference_output = inference_base / "pred/rgb_dem_extent"

# Create output directories
os.makedirs(s2_inference_output, exist_ok=True)
os.makedirs(rgbdem_inference_output, exist_ok=True)

# Hardcoded filenames (UPDATE THESE if using different files)
# Base filename without suffixes
BASE_FILENAME = "ortho_RGBN_dee_2024"  # UPDATE THIS for different datasets

# S2 Model files (10m resolution)
s2_img_file = inference_images_s2 / f"{BASE_FILENAME}_res_10m_cleaned_img.tif"
s2_lab_file = inference_labels_s2 / f"{BASE_FILENAME}_res_10m_cleaned_lab.tif"
s2_pred_file = s2_inference_output / f"{BASE_FILENAME}_res_10m_cleaned_img_pred.tif"

# RGB+DEM Model files (2m resolution)
rgbdem_img_file = inference_images_aerial / f"{BASE_FILENAME}_res_2m_cleaned_img.tif"
rgbdem_dem_file = inference_images_dem / f"{BASE_FILENAME}_res_10m_cleaned_img_dtm.tif"
rgbdem_lab_file = inference_labels_rgbdem / f"{BASE_FILENAME}_res_2m_cleaned_lab.tif"
rgbdem_pred_file = rgbdem_inference_output / f"{BASE_FILENAME}_res_2m_cleaned_img_pred.tif"

print(f"\n✓ Inference paths configured:")
print(f"\nCheckpoints:")
print(f"  S2: {s2_checkpoint_path}")
print(f"  RGB+DEM: {rgbdem_checkpoint_path}")
print(f"\nS2 Model Files (10m):")
print(f"  Image: {s2_img_file.name} - Exists: {s2_img_file.exists()}")
print(f"  Label: {s2_lab_file.name} - Exists: {s2_lab_file.exists()}")
print(f"  Prediction: {s2_pred_file.name} - Exists: {s2_pred_file.exists()}")
print(f"\nRGB+DEM Model Files (2m):")
print(f"  Image: {rgbdem_img_file.name} - Exists: {rgbdem_img_file.exists()}")
print(f"  DEM: {rgbdem_dem_file.name} - Exists: {rgbdem_dem_file.exists()}")
print(f"  Label: {rgbdem_lab_file.name} - Exists: {rgbdem_lab_file.exists()}")
print(f"  Prediction: {rgbdem_pred_file.name} - Exists: {rgbdem_pred_file.exists()}")

NameError: name 's2_checkpoint_local' is not defined

### 3.4 Run Inference with Sentinel-2 Model

Run prediction on inference images using the pre-trained S2 checkpoint.

For inference, we'll use the **config from HuggingFace** rather than the local config used for fine-tuning.

In [ ]:
# Use the downloaded config from HuggingFace
# Construct the prediction command
s2_predict_command = f"""terratorch predict \
    --config {s2_config_path} \
    --ckpt_path {s2_checkpoint_path} \
    --trainer.default_root_dir {output_dir} \
    --predict_output_dir {s2_inference_output} \
    --data.init_args.predict_data_root.S2L2A {inference_images_s2}"""

print("S2 Prediction command:")
print(s2_predict_command)
print(f"\nUsing config: {s2_config_path}")
print(f"Using checkpoint: {s2_checkpoint_path}")

In [ ]:
# Run S2 prediction
!{s2_predict_command}

### 3.5 Visualize S2 Results

Let's visualize the predictions from the S2 model alongside the input imagery and labels.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

# Set up for extent model (binary classification)
CLASS_COLORS = ['#d9d9d9', '#2c7bb6']  # Not Saltmarsh, Saltmarsh
CLASS_CMAP = ListedColormap(CLASS_COLORS)
CLASS_LABELS = ['Not Saltmarsh', 'Saltmarsh']
N_CLASSES = 2

In [ ]:
# Import visualization helper functions TODO- COLAB VERSION NEEDS TO PULL THIS FROM THE REPO
from helper import load_raster, normalize_rgb, plot_panel

print("✓ Visualization functions loaded from helper.py")

In [ ]:
# Load S2 data using hardcoded filenames from section 3.3
img_data = normalize_rgb(load_raster(s2_img_file, rgb_only=True))
lab_data = load_raster(s2_lab_file)
pred_data = load_raster(s2_pred_file)

# Create figure with 1 row × 3 columns
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('S2 Model Results (10m resolution)', fontsize=16, fontweight='bold')

# Plot panels
plot_panel(axes[0], img_data, 'Input S2 Imagery')
plot_panel(axes[1], lab_data, 'Ground Truth Labels', 
           cmap=CLASS_CMAP, is_categorical=True, n_classes=N_CLASSES)
plot_panel(axes[2], pred_data, 'Model Predictions', 
           cmap=CLASS_CMAP, is_categorical=True, n_classes=N_CLASSES)

# Add legend
legend_elements = [Patch(facecolor=CLASS_COLORS[i], label=CLASS_LABELS[i]) 
                   for i in range(len(CLASS_LABELS))]
fig.legend(handles=legend_elements, loc='lower center', ncol=2, 
           fontsize=11, frameon=True, bbox_to_anchor=(0.5, -0.05))

plt.tight_layout()
plt.show()

print(f"\n✓ Visualized S2 model results")
print(f"  Image: {s2_img_file.name}")
print(f"  Label: {s2_lab_file.name}")
print(f"  Prediction: {s2_pred_file.name}")

---

## 4. Multi-Modal Inference

### 4.1 Run Inference with RGB+DEM Model

Now let's run inference using a multi-modal model that combines aerial RGB imagery with Digital Elevation Model (DEM) data.

This model uses:
- **RGB**: High-resolution aerial imagery (2m resolution)
- **DEM**: Digital Elevation Model for terrain information

We'll use the **config from HuggingFace** for this model as well.

In [ ]:
# Use the downloaded config from HuggingFace
# Construct the prediction command for RGB+DEM model
# Note: RGB and DEM are in separate directories
rgbdem_predict_command = f"""terratorch predict \
    --config {rgbdem_config_path} \
    --ckpt_path {rgbdem_checkpoint_path} \
    --trainer.default_root_dir {output_dir} \
    --predict_output_dir {rgbdem_inference_output} \
    --data.init_args.predict_data_root.RGB {inference_images_aerial} \
    --data.init_args.predict_data_root.DEM {inference_images_dem}"""

print("RGB+DEM Prediction command:")
print(rgbdem_predict_command)
print(f"\nUsing config: {rgbdem_config_path}")
print(f"Using checkpoint: {rgbdem_checkpoint_path}")

In [ ]:
# Run RGB+DEM prediction
!{rgbdem_predict_command}

### 4.2 Compare S2 vs RGB+DEM Models

Let's compare the predictions from both models to understand the benefits of multi-modal approaches.

We'll create a 2-row comparison showing:
- **Row 1 (S2 Model)**: Input imagery, DEM (blank), Labels, Predictions
- **Row 2 (RGB+DEM Model)**: Input imagery, DEM, Labels, Predictions

In [ ]:
# Load data using hardcoded filenames from section 3.3

# Load S2 data (10m resolution)
s2_img_data = normalize_rgb(load_raster(s2_img_file, rgb_only=True))
s2_lab_data = load_raster(s2_lab_file)
s2_pred_data = load_raster(s2_pred_file)

# Load RGB+DEM data (2m resolution)
rgbdem_img_data = normalize_rgb(load_raster(rgbdem_img_file, rgb_only=True))
rgbdem_dem_data = load_raster(rgbdem_dem_file)
rgbdem_lab_data = load_raster(rgbdem_lab_file)
rgbdem_pred_data = load_raster(rgbdem_pred_file)

# Create figure with 2 rows × 4 columns
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('Model Comparison: S2 (10m) vs RGB+DEM (2m)', fontsize=16, fontweight='bold')

# Row 1: S2 Model (10m resolution)
plot_panel(axes[0, 0], s2_img_data, 'S2 - Input Imagery (10m)')
plot_panel(axes[0, 1], None, 'S2 - DEM (N/A)')  # Blank for S2
plot_panel(axes[0, 2], s2_lab_data, 'S2 - Labels', 
           cmap=CLASS_CMAP, is_categorical=True, n_classes=N_CLASSES)
plot_panel(axes[0, 3], s2_pred_data, 'S2 - Predictions', 
           cmap=CLASS_CMAP, is_categorical=True, n_classes=N_CLASSES)

# Row 2: RGB+DEM Model (2m resolution)
plot_panel(axes[1, 0], rgbdem_img_data, 'RGB+DEM - Input Imagery (2m)')
plot_panel(axes[1, 1], rgbdem_dem_data, 'RGB+DEM - DEM (10m)', is_dem=True)
plot_panel(axes[1, 2], rgbdem_lab_data, 'RGB+DEM - Labels', 
           cmap=CLASS_CMAP, is_categorical=True, n_classes=N_CLASSES)
plot_panel(axes[1, 3], rgbdem_pred_data, 'RGB+DEM - Predictions', 
           cmap=CLASS_CMAP, is_categorical=True, n_classes=N_CLASSES)

# Add legend
legend_elements = [Patch(facecolor=CLASS_COLORS[i], label=CLASS_LABELS[i]) 
                   for i in range(len(CLASS_LABELS))]
fig.legend(handles=legend_elements, loc='lower center', ncol=2, 
           fontsize=11, frameon=True, bbox_to_anchor=(0.5, -0.02))

plt.tight_layout()
plt.show()

print(f"\n✓ Comparison visualization complete")
print(f"\nS2 Model (10m):")
print(f"  Image: {s2_img_file.name}")
print(f"  Label: {s2_lab_file.name}")
print(f"  Prediction: {s2_pred_file.name}")
print(f"\nRGB+DEM Model (2m):")
print(f"  Image: {rgbdem_img_file.name}")
print(f"  DEM: {rgbdem_dem_file.name}")
print(f"  Label: {rgbdem_lab_file.name}")
print(f"  Prediction: {rgbdem_pred_file.name}")

### Key Insights: Multi-Modal vs Single-Modal

Comparing the two models reveals important differences:

**Multi-modal (RGB+DEM) advantages:**
- **Higher Resolution**: Aerial imagery at 2m vs Sentinel-2 at 10m provides finer spatial detail
- **Terrain Context**: DEM provides elevation information crucial for salt marsh identification
- **Complementary Information**: RGB captures visual features while DEM captures topographic features
- **Improved Accuracy**: Combining modalities typically improves detection performance in complex areas

**Single-modal (S2) advantages:**
- **Global Coverage**: Sentinel-2 provides consistent global coverage
- **Temporal Resolution**: Regular revisit times (5 days) for monitoring changes over time
- **Free and Open**: Publicly available data with no acquisition costs
- **Spectral Bands**: Multiple spectral bands beyond RGB (NIR, SWIR, etc.) for vegetation analysis
- **Scalability**: Easier to scale to large areas without custom aerial surveys

**When to use each:**
- Use **S2 models** for: Large-area mapping, temporal monitoring, global applications, budget-constrained projects
- Use **RGB+DEM models** for: High-precision mapping, detailed site assessments, areas with complex topography

In [ ]:
print("Model comparison visualization to be added here.")

### Key Insights: Multi-Modal vs Single-Modal

Multi-modal models (RGB+DEM) often provide advantages:

- **Higher Resolution**: Aerial imagery at 2m vs Sentinel-2 at 10m
- **Terrain Context**: DEM provides elevation information crucial for salt marsh identification
- **Complementary Information**: RGB captures visual features while DEM captures topographic features
- **Improved Accuracy**: Combining modalities typically improves detection performance

However, single-modal S2 models have benefits too:

- **Global Coverage**: Sentinel-2 provides consistent global coverage
- **Temporal Resolution**: Regular revisit times for monitoring changes
- **Free and Open**: Publicly available data
- **Spectral Bands**: Multiple spectral bands beyond RGB

---

## 📖 Next Steps

Continue to the next notebook:

- **1.2 Multimodal Inference with TerraMind** - Learn more about running inference with different data modalities

---

## 🔗 Additional Resources

- [TerraMind on HuggingFace](https://huggingface.co/ibm-esa-geospatial)
- [TerraTorch Documentation](https://github.com/IBM/terratorch)
- [TerraKit Documentation](https://torchgeo.org/terrakit/)
- [Prithvi Model Hub](https://huggingface.co/ibm-nasa-geospatial)

---

<!--
Copyright (c) 2026 IBM Corp

This software is released under the MIT License.
https://opensource.org/licenses/MIT
-->